# OptiCrop – Exploratory Data Analysis
Smart Agricultural Production Optimization Engine

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
%matplotlib inline
sns.set_theme(style='whitegrid')

df = pd.read_csv('../Crop_recommendation.csv')
print('Shape:', df.shape)
df.head()

In [ ]:
# Basic statistics
df.describe().round(2)

In [ ]:
# Missing values
print('Missing values:\n', df.isnull().sum())
print('\nCrop counts:\n', df['label'].value_counts())

In [ ]:
# Crop distribution
plt.figure(figsize=(14,5))
df['label'].value_counts().plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Crop Distribution')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(9,7))
sns.heatmap(df.drop('label',axis=1).corr(), annot=True, fmt='.2f', cmap='YlGn')
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
# Feature distributions
features = ['N','P','K','temperature','humidity','ph','rainfall']
fig, axes = plt.subplots(2, 4, figsize=(16,8))
for i, feat in enumerate(features):
    axes.flatten()[i].hist(df[feat], bins=30, color='#2e7d32', edgecolor='black')
    axes.flatten()[i].set_title(feat)
axes.flatten()[-1].axis('off')
plt.suptitle('Feature Distributions')
plt.tight_layout()
plt.show()

In [ ]:
# NPK per crop
df.groupby('label')[['N','P','K']].mean().plot(kind='bar', figsize=(14,5), colormap='Set2', edgecolor='black')
plt.title('Average NPK per Crop')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Pairplot
sample = df.groupby('label').apply(lambda x: x.sample(min(10,len(x)))).reset_index(drop=True)
sns.pairplot(sample, hue='label', vars=['N','P','K','temperature'], plot_kws={'alpha':0.6}, height=2.2)
plt.suptitle('Pairplot of Key Features', y=1.02)
plt.show()

In [ ]:
# Scipy: ANOVA test – is temperature significantly different across crops?
groups = [group['temperature'].values for _, group in df.groupby('label')]
f_stat, p_val = stats.f_oneway(*groups)
print(f'ANOVA F-statistic: {f_stat:.2f}, p-value: {p_val:.4e}')
print('Temperature differs significantly across crops!' if p_val < 0.05 else 'No significant difference.')

In [ ]:
# Model training summary
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report

le = LabelEncoder()
X = df[features].values
y = le.fit_transform(df['label'])

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

print(f'Test Accuracy: {accuracy_score(y_test, rf.predict(X_test)):.4f}')
cv = cross_val_score(rf, X_scaled, y, cv=5)
print(f'5-Fold CV: {cv.mean():.4f} ± {cv.std():.4f}')